In [1]:

import csv
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
HEADERS = {
    "User-Agent": "DSC2107-Group9-Course-Project (educational exercise)"
}
OUTPUT = Path("data/raw")
OUTPUT.mkdir(parents=True, exist_ok=True)


def get_page(url):
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def scrape_books():
    results = []
    page_url = BASE_URL

    while page_url:
        soup = get_page(page_url)

        for book in soup.select("article.product_pod"):
            title = book.h3.a.get("title", "").strip()
            relative_url = book.h3.a.get("href", "")
            product_url = urljoin(page_url, relative_url)

            price_tag = book.select_one(".price_color")
            availability_tag = book.select_one(".availability")
            rating_tag = book.select_one("p.star-rating")

            results.append({
                "title": title,
                "price": price_tag.get_text(strip=True)
                         if price_tag else None,
                "availability": availability_tag.get_text(" ", strip=True)
                                if availability_tag else None,
                "rating": (
                    rating_tag.get("class", [])[-1]
                    if rating_tag and len(rating_tag.get("class", [])) > 1
                    else None
                ),
                "product_url": product_url
            })

        next_button = soup.select_one("li.next a")
        page_url = (
            urljoin(page_url, next_button["href"])
            if next_button else None
        )

        print(f"Books collected so far: {len(results)}")

        if page_url:
            time.sleep(3)

    # Save raw practice results as CSV
    output_file = OUTPUT / "books_practice.csv"

    with output_file.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    print(f"Finished: {len(results)} books")
    print(f"Saved to {output_file}")


if __name__ == "__main__":
    scrape_books()


Books collected so far: 20


ConnectionError: HTTPSConnectionPool(host='books.toscrape.com', port=443): Read timed out.